# Weaving


In [ ]:
import math
import numpy as np
from numpy.typing import NDArray

arr = np.array

## Structure

Axes:

- $V$: lengthwise direction of warp, bottom to top, texture horizontal
- $U$: sidewise direction of weft, left to right, texture vertical
- $W$: facewise direction, back to front, texture bump/normal

Cell/knot types:

- `│`: warp-facing ($w \gt 0$)
- `─`: weft-facing ($w \le 0$)

Everything is front/back symmetric with $w' = -w$ with tension effects $± \delta w_0$

Knots are placed in a square/rectangular grid, types are defined by design of a pattern.


## ASCII art


In [ ]:
def picture(pattern: NDArray, vsize=None, usize=None):
    """Depict weaving pattern in `|` and `-` characters"""
    vdim, udim = pattern.shape
    vsize = vsize or vdim + 1
    usize = usize or udim + 1

    def sample(v, u):
        return pattern[v % vdim, u % udim]

    return "\n".join("".join("|" if sample(v, u) > 0 else "-" for u in range(usize)) for v in range(vsize))

In [ ]:
print(picture(arr([[+1, -1], [-1, +1]])))

## Patterns


#### Plain

Simple checkerboard

```
│─│─│─│─
─│─│─│─│
│─│─│─│─
─│─│─│─│
```

$$
w = \begin{cases}
+1, (u - v) \mod 2 = 0 \\
-1, \text{else}
\end{cases}
$$


In [ ]:
def gen_plain(u: int, v: int):
    return +1 if (u - v) % 2 == 0 else -1

In [ ]:
plain = np.array([[gen_plain(u, v) for u in (0, 1)] for v in (0, 1)])

#### Twill

Sequence of various numbers of │ and ─

3/1 Z1

```
│││─│││─
││─│││─│
│─│││─││
─│││─│││
```

1/5 Z2

```
|-----|
----|--
--|----
|-----|
```

Parameters:

- $a$: number of warp faced
- $b$: number of weft faced
- $d$: shift of the pattern (usually ±1)
- $p = a + b$: weft period of the pattern
- $l = LCM(p, d) / d$: warp period

$$
w = \begin{cases}
+1, (u - v \cdot d) \mod p < a \\
-1, \text{else}
\end{cases}
$$

The textile twill formula counts rows from bottom to top, so the $d$ is inverted


In [ ]:
def gen_twill(a: int, b: int, d: int):
    p = a + b

    def gen(u: int, v: int):
        return +1 if (u + v * d) % p < a else -1

    return gen

In [ ]:
def np_twill(a: int, b: int, d: int):
    udim = a + b
    vdim = math.lcm(udim, abs(d)) // abs(d)
    gen = gen_twill(a, b, d)

    return np.array([[gen(u, v) for u in range(udim)] for v in range(vdim)])

In [ ]:
denim = np_twill(3, 1, 1)
satin5 = np_twill(1, 5, 2)
tartan = np_twill(2, 2, -1)

#### Waffle

Patterns with long loose runs

```
│─│─│─│
││─│─││
│││─│││
││─│─││
│─│─│─│
─│───│─
│─────│
─│───│─
│─│─│─│
```

The tension forms layered structure with multiple values of $w$

Shift is assymetrical and weft = warp × -1 doesn't quite work


In [ ]:
waffle = np.array([
    [+1, -1, +1, -1, +1, -1],  # │─│─│─
    [+1, +1, -1, +1, -1, +1],  # ││─│─│
    [+1, +1, +1, -1, +1, +1],  # │││─││
    [+1, +1, -1, +1, -1, +1],  # ││─│─│
    [+1, -1, +1, -1, +1, -1],  # │─│─│─
    [-1, +1, -1, -1, -1, +1],  # ─│───│
    [+1, -1, -1, -1, -1, -1],  # │─────
    [-1, +1, -1, -1, -1, +1],  # ─│───│
])

In [ ]:
print(picture(waffle, 9, 13))

### Weaving Drafts


In [ ]:
from weaving_draft import WeavingDraft

In [ ]:
# minimal waffle from https://gagnagrunnur.textilmidstod.is
gagnagrunnur = WeavingDraft.parse(
    tieup="""
    x...
    .x..
    x.x.
    xx.x
    """,
    threading="""
    ..x...
    .x.x..
    x...x.
    .....x
    """,
    treadling="""
    ..x.
    .x..
    x...
    .x..
    ..x.
    ...x
    """,
)

In [ ]:
print(picture(gagnagrunnur.compile(), 13, 13))

---


In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
import pyvista as pv
from matplotlib.cm import tab10

# pv.set_jupyter_backend("server")
tab10colors = tab10.colors  # type: ignore
warpclr = tab10.colors[3]
weftclr = tab10.colors[2]

## Topology


In [ ]:
# pattern = gagnagrunnur.compile() * 2 - 1
# pattern = tartan
pattern = waffle
usize = 16
vsize = 16
thick = 0.5

In [ ]:
pattern

In [ ]:
vdim, udim = pattern.shape
size = usize * vsize
V, U = np.mgrid[0:vsize, 0:usize]
pointgrid = np.array([U, V, pattern[V % vdim, U % udim]], dtype=np.float32).T
pointdump = pointgrid.reshape((size, 3))

In [ ]:
warplines = arr([[vsize, *range(u, u + size, usize)] for u in range(usize)]).flatten()
weftlines = arr([[usize, *range(v * usize, (v + 1) * usize)] for v in range(vsize)]).flatten()
verts = arr([np.ones((size,)), np.arange(size)], dtype=np.int_).T.flatten()
pvwarp = pv.PolyData(pointdump * arr([1, 1, 0.5 * thick]), lines=warplines.flatten(), verts=verts)
pvweft = pv.PolyData(pointdump * arr([1, 1, -0.5 * thick]), lines=weftlines.flatten(), verts=verts)

In [ ]:
plot = pv.Plotter()
plot.set_background(pv.Color("#303030"))
plot.add_mesh(pvwarp, color=warpclr, line_width=8, point_size=10, label="warp", render_points_as_spheres=True)
plot.add_mesh(pvweft, color=weftclr, line_width=8, point_size=10, label="weft", render_points_as_spheres=True)
plot.view_xy()
plot.add_legend()
plot.show_grid()
plot.show()

## Curves

Just smothing topology anchors with splines


In [ ]:
from splines import M2, M3, mega_curve


def curve(points):
    return mega_curve(3, M3, points, np.arange(0.0, 1.0, 0.125))

In [ ]:
warpcurves = [curve(pointgrid[:, u] * arr([1.0, 1.0, 0.5 * thick])) for u in range(2, usize - 2)]
weftcurves = [curve(pointgrid[v, :] * arr([1.0, 1.0, -0.5 * thick])) for v in range(2, vsize - 2)]

In [ ]:
plot = pv.Plotter()
plot.set_background(pv.Color("#303030"))
pva_warps = plot.add_mesh(pvwarp, color=warpclr, line_width=2, point_size=4, label="warp", render_points_as_spheres=True)
pva_wefts = plot.add_mesh(pvweft, color=weftclr, line_width=2, point_size=4, label="weft", render_points_as_spheres=True)
plot.show()

In [ ]:
def toggle_grid(flag):
    pva_warps.SetVisibility(flag)
    pva_wefts.SetVisibility(flag)


plot.add_checkbox_button_widget(toggle_grid, value=True, size=32, position=(8, 8))

In [ ]:
pvsplines = (
    [pv.Spline(crv).tube(radius=0.75 * thick, n_sides=12) for crv in warpcurves],
    [pv.Spline(crv).tube(radius=0.7 * thick, n_sides=12) for crv in weftcurves],
)
pvactors = (
    [plot.add_mesh(spl, color=warpclr, smooth_shading=True) for spl in pvsplines[0]],
    [plot.add_mesh(spl, color=weftclr, smooth_shading=True) for spl in pvsplines[1]],
)

In [ ]:
def toggle_color(flag):
    color1 = warpclr if flag else (0.5, 0.5, 0.5)
    color2 = weftclr if flag else (0.5, 0.5, 0.5)

    for pva in pvactors[0]:
        pva.GetProperty().SetColor(color1)
    for pva in pvactors[1]:
        pva.GetProperty().SetColor(color2)


plot.add_checkbox_button_widget(toggle_color, value=True, size=32, position=(8, 8 + 32 + 8))

In [ ]:
def clip(w):
    bounds = w.bounds
    for pvm, pva in zip(pvsplines[0], pvactors[0]):
        pva.mapper.dataset = pvm.clip_box(bounds, invert=False)
    for pvm, pva in zip(pvsplines[1], pvactors[1]):
        pva.mapper.dataset = pvm.clip_box(bounds, invert=False)


plot.add_box_widget(clip)